# Week 9 · Topic 2 — Multi-Head Attention & Positional Encoding
### Demo Notebook: Splitting Attention Into Heads, Then Adding a Sense of Order (Pure Python + NumPy)

**Goal:** extend Topic 1's single attention mechanism into **multiple heads** running in parallel,
then solve the problem Topic 1 quietly left open — attention has no idea what order the words
came in. We fix that with **positional encoding**.

This notebook reuses the exact `scaled_dot_product_attention` logic from Topic 1's notebook —
nothing about the core attention math changes here, we're only changing *how many times* we run
it and *what* we feed into it.

Four parts:
1. **Recap** — bring in Topic 1's attention function
2. **Multi-head attention** — split, attend per head, concatenate, project
3. **Positional encoding** — the sine/cosine formula, built from scratch
4. **Putting it together** — position-aware multi-head attention on the full running example

Running example (same as Topic 1): **English** `"I am going to the market"` /
**Yoruba** `"Mo n lọ si ọja"`.

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
print("NumPy ready.")


NumPy ready.


## Part 1 — Recap: Reusing Topic 1's Attention Function

Before building anything new, we bring back the exact `softmax_rows` and
`scaled_dot_product_attention` functions from Topic 1's notebook. Multi-head attention doesn't
change this math at all — it just calls this same function once per head, on a smaller slice of
the data each time.

In [ ]:
# Reused from Topic 1's notebook, unchanged.
def softmax_rows(x):
    x = np.array(x, dtype=float)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

def scaled_dot_product_attention(X, W_Q, W_K, W_V):
    """Same function from Topic 1: Score -> Scale -> Normalise -> Blend."""
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    d_k = Q.shape[-1]
    raw_scores = Q @ K.T
    scaled_scores = raw_scores / np.sqrt(d_k)
    weights = softmax_rows(scaled_scores)
    output = weights @ V
    return output, weights

print("Reused Topic 1 functions are ready: softmax_rows(), scaled_dot_product_attention()")


Reused Topic 1 functions are ready: softmax_rows(), scaled_dot_product_attention()


In [ ]:
# The running example, same tokens as Topic 1. This time we give each token a
# LARGER embedding (8 numbers instead of 4) so we have room to split it across 2 heads.
tokens_full = ["I", "am", "going", "to", "the", "market"]
yoruba_reference = "Mo n lọ si ọja"

d_model = 8
np.random.seed(7)
embeddings_full = {t: np.round(np.random.randn(d_model), 2) for t in tokens_full}
X_full = np.stack([embeddings_full[t] for t in tokens_full])

print("English tokens:", tokens_full)
print("Yoruba translation (reference only):", yoruba_reference)
print("Embedding matrix shape:", X_full.shape, "(6 tokens x 8 dimensions)")


English tokens: ['I', 'am', 'going', 'to', 'the', 'market']
Yoruba translation (reference only): Mo n lọ si ọja
Embedding matrix shape: (6, 8) (6 tokens x 8 dimensions)


## Part 2 — Multi-Head Attention

**The idea (Slide 4):** instead of running attention once with the full embedding, we split
Query, Key, and Value into smaller equal-sized chunks — one chunk per head — run attention
independently on each chunk, then join the results back together.

We'll use **2 heads**. With an 8-dimensional embedding, that means each head works with a
4-dimensional slice.

In [ ]:
# Step 1 — project the full embeddings into full-size Query, Key, Value matrices,
# exactly like Topic 1, just with bigger weight matrices (8x8 instead of 4x4).
np.random.seed(11)
W_Q = np.round(np.random.randn(d_model, d_model) * 0.3, 2)
W_K = np.round(np.random.randn(d_model, d_model) * 0.3, 2)
W_V = np.round(np.random.randn(d_model, d_model) * 0.3, 2)
W_O = np.round(np.random.randn(d_model, d_model) * 0.3, 2)   # final output projection (new this topic)

Q_full = X_full @ W_Q
K_full = X_full @ W_K
V_full = X_full @ W_V

print("Q_full shape:", Q_full.shape, "(6 tokens x 8 dimensions, before splitting into heads)")


Q_full shape: (6, 8) (6 tokens x 8 dimensions, before splitting into heads)


In [ ]:
# Step 2 — split Q, K, V into per-head slices (Slide 5).
# 8 dimensions split across 2 heads = 4 dimensions per head.
num_heads = 2
head_dim = d_model // num_heads
print(f"num_heads = {num_heads}, head_dim = {head_dim}  (must divide evenly: {d_model} / {num_heads} = {head_dim})")

Q_heads = [Q_full[:, h*head_dim:(h+1)*head_dim] for h in range(num_heads)]
K_heads = [K_full[:, h*head_dim:(h+1)*head_dim] for h in range(num_heads)]
V_heads = [V_full[:, h*head_dim:(h+1)*head_dim] for h in range(num_heads)]

for h in range(num_heads):
    print(f"Head {h}: Q slice shape {Q_heads[h].shape}, K slice shape {K_heads[h].shape}, V slice shape {V_heads[h].shape}")


num_heads = 2, head_dim = 4  (must divide evenly: 8 / 2 = 4)
Head 0: Q slice shape (6, 4), K slice shape (6, 4), V slice shape (6, 4)
Head 1: Q slice shape (6, 4), K slice shape (6, 4), V slice shape (6, 4)


In [ ]:
# Step 3 — run attention independently in each head (Slide 6).
# Each head gets its own Score -> Scale -> Normalise -> Blend, reusing the SAME
# scaled_dot_product_attention logic from Topic 1 -- just called on Q/K/V slices
# instead of the full-size Q/K/V, and without re-projecting (already projected above).
def attention_from_qkv(Q, K, V):
    """Same four steps as scaled_dot_product_attention, but starting from
    already-projected Q, K, V slices rather than raw embeddings."""
    d_k = Q.shape[-1]
    raw_scores = Q @ K.T
    scaled_scores = raw_scores / np.sqrt(d_k)
    weights = softmax_rows(scaled_scores)
    output = weights @ V
    return output, weights

head_outputs = []
head_weights = []
for h in range(num_heads):
    out_h, w_h = attention_from_qkv(Q_heads[h], K_heads[h], V_heads[h])
    head_outputs.append(out_h)
    head_weights.append(w_h)

print("Head 0 attention weights (rows = Query word, columns = Key word):")
print(head_weights[0])
print()
print("Head 1 attention weights:")
print(head_weights[1])


Head 0 attention weights (rows = Query word, columns = Key word):
[[0.196 0.234 0.023 0.128 0.363 0.056]
 [0.227 0.212 0.057 0.166 0.23  0.108]
 [0.343 0.26  0.152 0.152 0.026 0.067]
 [0.132 0.114 0.242 0.132 0.157 0.222]
 [0.007 0.005 0.489 0.044 0.02  0.435]
 [0.333 0.326 0.091 0.135 0.065 0.049]]

Head 1 attention weights:
[[0.272 0.185 0.065 0.039 0.385 0.053]
 [0.145 0.221 0.19  0.133 0.227 0.084]
 [0.031 0.039 0.139 0.39  0.006 0.394]
 [0.051 0.031 0.082 0.249 0.007 0.58 ]
 [0.059 0.101 0.061 0.024 0.746 0.008]
 [0.061 0.048 0.144 0.239 0.027 0.481]]


**Notice:** Head 0 and Head 1 produce *different* attention patterns even though they
started from the same sentence — each head learned (via its own random projection here) to
focus on a different slice of the embedding, so each ends up "noticing" different relationships.
This is exactly Slide 3's point: different heads can specialise in different kinds of
relationships.

In [ ]:
# Step 4 — concatenate the heads back together, then apply the final projection (Slide 7).
concat = np.concatenate(head_outputs, axis=-1)
print("Concatenated shape:", concat.shape, "(back to 6 tokens x 8 dimensions)")

mha_output = concat @ W_O
print("\nMulti-head attention output (after final projection):")
for tok, vec in zip(tokens_full, mha_output):
    print(f"  {tok:8s} -> {vec}")


Concatenated shape: (6, 8) (back to 6 tokens x 8 dimensions)

Multi-head attention output (after final projection):
  I        -> [-0.196 -0.326  0.058 -0.135  0.154  0.198 -0.206 -0.106]
  am       -> [-0.202 -0.058 -0.184 -0.017  0.101  0.338 -0.03  -0.261]
  going    -> [-0.286  0.21  -0.653  0.367  0.353  0.398 -0.238 -0.55 ]
  to       -> [-0.316 -0.01  -0.328  0.42   0.139  0.512 -0.065 -0.315]
  the      -> [-0.713 -1.285  1.187  0.086 -0.636  1.002  0.502  0.682]
  market   -> [-0.296  0.095 -0.609  0.354  0.448  0.309 -0.397 -0.514]


**Check:** the output shape matches the input shape (6 tokens × 8 dimensions) — multi-head
attention transforms the representation but doesn't change its size, which is exactly what lets
it be stacked or combined with other pieces later (Topic 4).

## Part 3 — Positional Encoding

**The problem (Slide 9):** nothing in the attention math above depends on *where* each word sits
in the sentence — only on its content. Scramble the six words and the exact same code would
produce identical attention patterns. We need to explicitly inject position information.

**The fix (Slides 11–12):** add a sine/cosine pattern to each token's embedding — a unique
"fingerprint" per position.

In [ ]:
# Implement the sinusoidal positional encoding formula from scratch:
#   PE(pos, 2i)   = sin(pos / 10000^(2i / d_model))
#   PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))
def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, None]                       # shape (seq_len, 1)
    i = np.arange(d_model)                                       # 0, 1, 2, ..., d_model-1
    div_term = np.power(10000, (2 * (i // 2)) / d_model)          # shared denominator per dimension pair
    angles = position / div_term                                 # shape (seq_len, d_model)
    pe[:, 0::2] = np.sin(angles[:, 0::2])   # even dimensions -> sine
    pe[:, 1::2] = np.cos(angles[:, 1::2])   # odd dimensions  -> cosine
    return pe

seq_len = len(tokens_full)
PE = positional_encoding(seq_len, d_model)

print("Positional encoding matrix shape:", PE.shape, "(6 positions x 8 dimensions)")
print(PE)


Positional encoding matrix shape: (6, 8) (6 positions x 8 dimensions)
[[ 0.     1.     0.     1.     0.     1.     0.     1.   ]
 [ 0.841  0.54   0.1    0.995  0.01   1.     0.001  1.   ]
 [ 0.909 -0.416  0.199  0.98   0.02   1.     0.002  1.   ]
 [ 0.141 -0.99   0.296  0.955  0.03   1.     0.003  1.   ]
 [-0.757 -0.654  0.389  0.921  0.04   0.999  0.004  1.   ]
 [-0.959  0.284  0.479  0.878  0.05   0.999  0.005  1.   ]]


**Reading the matrix:** each row is one position's unique "fingerprint." Look at the
right-most columns (higher dimension pairs) — they change very slowly from row to row, while the
left-most columns change quickly. That's the low-frequency / high-frequency mix described on
Slide 12: fast-oscillating dimensions capture fine local position differences, slow-oscillating
ones capture broader position differences.

In [ ]:
# Confirm every position gets a distinct pattern (no two rows identical).
distinct_rows = len(set(tuple(np.round(row, 3)) for row in PE))
print(f"{distinct_rows} distinct positional patterns out of {seq_len} positions.")


6 distinct positional patterns out of 6 positions.


## Part 4 — Putting It Together: Position-Aware Multi-Head Attention

**Slide 13:** positional encoding is simply *added* to the token embedding, element-by-element,
before anything else happens. That combined vector is what gets split into Query, Key, Value.

In [ ]:
# Add positional encoding directly to the token embeddings (Slide 13).
X_pos = X_full + PE

print("Original embedding for 'I' (position 0):      ", X_full[0])
print("Positional encoding at position 0:             ", PE[0])
print("Combined (position-aware) embedding for 'I':   ", X_pos[0])


Original embedding for 'I' (position 0):       [ 1.69 -0.47  0.03  0.41 -0.79  0.   -0.   -1.75]
Positional encoding at position 0:              [0. 1. 0. 1. 0. 1. 0. 1.]
Combined (position-aware) embedding for 'I':    [ 1.69  0.53  0.03  1.41 -0.79  1.    0.   -0.75]


In [ ]:
# Wrap the full pipeline into one reusable function -- this is what Topic 4's
# notebook will import and reuse when assembling the complete Transformer block.
def multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads):
    """
    Runs multi-head self-attention on a sequence of (position-aware) embeddings.

    X : (seq_len, d_model) matrix -- token embeddings, optionally with positional
        encoding already added
    W_Q, W_K, W_V, W_O : (d_model, d_model) projection matrices
    num_heads : number of attention heads; d_model must be divisible by num_heads

    Returns:
        output : (seq_len, d_model) combined multi-head attention output
        all_weights : list of per-head (seq_len, seq_len) attention weight matrices
    """
    d_model = X.shape[-1]
    head_dim = d_model // num_heads

    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    outputs, all_weights = [], []
    for h in range(num_heads):
        sl = slice(h * head_dim, (h + 1) * head_dim)
        out_h, w_h = attention_from_qkv(Q[:, sl], K[:, sl], V[:, sl])
        outputs.append(out_h)
        all_weights.append(w_h)

    concat = np.concatenate(outputs, axis=-1)
    output = concat @ W_O
    return output, all_weights

print("multi_head_attention() is defined and ready to reuse in Topic 4.")


multi_head_attention() is defined and ready to reuse in Topic 4.


In [ ]:
# Run the full pipeline WITH positional encoding on the full running example.
mha_output_with_pos, weights_with_pos = multi_head_attention(X_pos, W_Q, W_K, W_V, W_O, num_heads)

print("Multi-head attention output, WITH positional encoding:")
for tok, vec in zip(tokens_full, mha_output_with_pos):
    print(f"  {tok:8s} -> {vec}")


Multi-head attention output, WITH positional encoding:
  I        -> [-0.511 -0.386 -0.03  -0.359  0.009  0.352  0.346 -0.08 ]
  am       -> [-0.488 -0.008 -0.447 -0.274  0.076  0.432  0.458 -0.372]
  going    -> [-0.403  0.323 -1.005 -0.014  0.26   0.326  0.211 -0.6  ]
  to       -> [-0.396  0.081 -0.504  0.12  -0.007  0.415  0.391 -0.265]
  the      -> [-0.725 -1.041  0.993 -0.215 -0.77   0.864  1.061  0.662]
  market   -> [-0.456  0.16  -0.729 -0.032  0.089  0.453  0.396 -0.445]


In [ ]:
# Compare against running the SAME pipeline WITHOUT positional encoding
# (i.e. on the raw embeddings from Part 2), to see what position information changed.
mha_output_no_pos, _ = multi_head_attention(X_full, W_Q, W_K, W_V, W_O, num_heads)

diff = np.abs(mha_output_with_pos - mha_output_no_pos)
print("Mean absolute difference introduced by positional encoding:", round(diff.mean(), 4))
print()
print("Per-token difference:")
for tok, d in zip(tokens_full, diff):
    print(f"  {tok:8s} -> mean abs diff {d.mean():.4f}")


Mean absolute difference introduced by positional encoding: 0.2054

Per-token difference:
  I        -> mean abs diff 0.1954
  am       -> mean abs diff 0.1969
  going    -> mean abs diff 0.2034
  to       -> mean abs diff 0.1748
  the      -> mean abs diff 0.2002
  market   -> mean abs diff 0.2619


**What this shows:** every word's output changed once positional encoding was added — even
though the *content* of each word's embedding didn't change, *where* it sits in the sentence now
measurably affects the result. This directly addresses Slide 10's problem: the same word
appearing at a different position would now produce a different output, because a different
positional pattern gets added to it.

## Wrap-Up

In this notebook we:

- Reused Topic 1's `scaled_dot_product_attention` logic unchanged — multi-head attention only
  changes *how many times* and *on what slices* we run it, not the underlying math
- Split Query, Key, and Value into 2 heads, ran attention independently in each, and saw the
  heads produce different attention patterns from the same sentence
- Concatenated the head outputs and applied a final linear projection to combine them
- Implemented the sinusoidal positional encoding formula from scratch and confirmed every
  position gets a unique pattern
- Added positional encoding directly to the token embeddings, and confirmed it measurably
  changes the attention output — solving the "no sense of order" gap from Part 2
- Packaged everything into one reusable `multi_head_attention()` function

**Up next (Topic 3):** a conceptual look at how this multi-head, position-aware attention fits
into the full Encoder-Decoder architecture — no new notebook there, since every mechanism it
describes is exactly what we just built.

**Then (Topic 4):** we'll reuse `multi_head_attention()` from this notebook, add a feed-forward
network, wrap both in residual connections and layer normalisation, and assemble the complete
Transformer block.